In [1]:
from collections import Counter
from pathlib import Path
import pandas as pd

import yaml

SPECIAL_TOKENS = {"<unk>", "<s>", "</s>", "<pad>"}

def load_marian_vocab(vocab_path, exclude_special_tokens=False):
    with open(vocab_path, encoding="utf-8") as f:
        vocab_data = yaml.safe_load(f)

    vocab = set(vocab_data.keys())
    if exclude_special_tokens:
        vocab -= SPECIAL_TOKENS
    return vocab

def calc_unk_stats(corpus_path, vocab_path, exclude_special_tokens=False):
    vocab = load_marian_vocab(vocab_path, exclude_special_tokens=exclude_special_tokens)

    with open(corpus_path, encoding="utf-8") as f:
        tokens = f.read().split()

    token_counter = Counter(tokens)
    unk_counter = Counter(token for token in tokens if token not in vocab)

    total_tokens = len(tokens)
    total_types = len(token_counter)
    unk_token_count = sum(unk_counter.values())
    unk_type_count = len(unk_counter)

    return {
        "total_tokens": total_tokens,
        "total_types": total_types,
        "vocab_size": len(vocab),
        "unk_token_count": unk_token_count,
        "unk_token_rate": unk_token_count / total_tokens if total_tokens else 0.0,
        "unk_type_count": unk_type_count,
        "unk_type_rate": unk_type_count / total_types if total_types else 0.0,
        "top_unk_tokens": unk_counter.most_common(20),
        "unk_tokens": unk_counter,
    }

def summarize_unk_for_sizes(sizes, lang="en", exclude_special_tokens=False):
    results = {}
    summary_rows = []

    for size in sizes:
        corpus_path = Path(f"data/test{size}k.{lang}")
        vocab_path = Path(f"vocabs/vocab.{lang}{size}k.yml")

        if not corpus_path.exists() or not vocab_path.exists():
            print(f"skip: missing file for {size}k ({corpus_path}, {vocab_path})")
            continue

        stats = calc_unk_stats(
            corpus_path,
            vocab_path,
            exclude_special_tokens=exclude_special_tokens,
        )
        results[size] = stats

        summary_rows.append({
            "vocab_size_k": size,
            "total_tokens": stats["total_tokens"],
            "total_types": stats["total_types"],
            "vocab_size": stats["vocab_size"],
            "unk_token_count": stats["unk_token_count"],
            "unk_token_rate": stats["unk_token_rate"],
            "unk_type_count": stats["unk_type_count"],
            "unk_type_rate": stats["unk_type_rate"],
        })

        print(f"\n===== {lang} {size}k =====")
        print(f"corpus         : {corpus_path}")
        print(f"vocab          : {vocab_path}")
        print(f"total tokens   : {stats['total_tokens']}")
        print(f"total types    : {stats['total_types']}")
        print(f"vocab size     : {stats['vocab_size']}")
        print(f"UNK tokens     : {stats['unk_token_count']}")
        print(f"UNK token rate : {stats['unk_token_rate']:.4%}")
        print(f"UNK types      : {stats['unk_type_count']}")
        print(f"UNK type rate  : {stats['unk_type_rate']:.4%}")
        print("top UNK tokens :", stats["top_unk_tokens"])

    summary_df = pd.DataFrame(summary_rows).set_index("vocab_size_k")
    if not summary_df.empty:
        summary_df[["unk_token_rate", "unk_type_rate"]] = (
            summary_df[["unk_token_rate", "unk_type_rate"]] * 100
        ).round(2)
        summary_df = summary_df.rename(columns={
            "unk_token_rate": "unk_token_rate_%",
            "unk_type_rate": "unk_type_rate_%",
        })

    return results, summary_df

sizes = [1, 2, 4, 6, 8, 10, 12, 14]
unk_results_en, unk_summary_en = summarize_unk_for_sizes(sizes, lang="en")
unk_summary_en



===== en 1k =====
corpus         : data/test1k.en
vocab          : vocabs/vocab.en1k.yml
total tokens   : 6452
total types    : 840
vocab size     : 999
UNK tokens     : 26
UNK token rate : 0.4030%
UNK types      : 8
UNK type rate  : 0.9524%
top UNK tokens : [('on', 15), ('0', 3), ('1', 2), ('2', 2), ('5', 1), ('▁eve', 1), ('1975', 1), ('3', 1)]

===== en 2k =====
corpus         : data/test2k.en
vocab          : vocabs/vocab.en2k.yml
total tokens   : 5545
total types    : 1192
vocab size     : 1918
UNK tokens     : 18
UNK token rate : 0.3246%
UNK types      : 8
UNK type rate  : 0.6711%
top UNK tokens : [('on', 7), ('0', 3), ('1', 2), ('2', 2), ('5', 1), ('1975', 1), ('3', 1), ('no', 1)]

===== en 4k =====
corpus         : data/test4k.en
vocab          : vocabs/vocab.en4k.yml
total tokens   : 5004
total types    : 1342
vocab size     : 3628
UNK tokens     : 17
UNK token rate : 0.3397%
UNK types      : 11
UNK type rate  : 0.8197%
top UNK tokens : [('0', 3), ('on', 3), ('1', 2), ('2', 2)

,total_tokens,total_types,vocab_size,unk_token_count,unk_token_rate_%,unk_type_count,unk_type_rate_%
vocab_size_k,,,,,,,
1,6452,840,999,26,0.40,8,0.95
2,5545,1192,1918,18,0.32,8,0.67
4,5004,1342,3628,17,0.34,11,0.82
6,4828,1334,4897,35,0.72,29,2.17
8,4714,1313,5682,154,3.27,116,8.83
10,4675,1294,5551,255,5.45,185,14.30
12,4664,1298,5551,247,5.30,190,14.64
14,4654,1297,5551,240,5.16,189,14.57


In [2]:
sizes = [1, 2, 4, 6, 8, 10, 12, 14]
unk_results_ku, unk_summary_ku = summarize_unk_for_sizes(sizes, lang="ku")
unk_summary_ku


===== ku 1k =====
corpus         : data/test1k.ku
vocab          : vocabs/vocab.ku1k.yml
total tokens   : 6015
total types    : 866
vocab size     : 998
UNK tokens     : 28
UNK token rate : 0.4655%
UNK types      : 8
UNK type rate  : 0.9238%
top UNK tokens : [('on', 15), ('no', 7), ('5', 1), ('1', 1), ('0', 1), ('1975', 1), ('2', 1), ('3', 1)]

===== ku 2k =====
corpus         : data/test2k.ku
vocab          : vocabs/vocab.ku2k.yml
total tokens   : 5262
total types    : 1268
vocab size     : 1989
UNK tokens     : 23
UNK token rate : 0.4371%
UNK types      : 9
UNK type rate  : 0.7098%
top UNK tokens : [('on', 10), ('no', 6), ('5', 1), ('1', 1), ('0', 1), ('aada', 1), ('1975', 1), ('2', 1), ('3', 1)]

===== ku 4k =====
corpus         : data/test4k.ku
vocab          : vocabs/vocab.ku4k.yml
total tokens   : 4691
total types    : 1549
vocab size     : 3845
UNK tokens     : 21
UNK token rate : 0.4477%
UNK types      : 11
UNK type rate  : 0.7101%
top UNK tokens : [('no', 6), ('on', 6), ('lee

,total_tokens,total_types,vocab_size,unk_token_count,unk_token_rate_%,unk_type_count,unk_type_rate_%
vocab_size_k,,,,,,,
1,6015,866,998,28,0.47,8,0.92
2,5262,1268,1989,23,0.44,9,0.71
4,4691,1549,3845,21,0.45,11,0.71
6,4477,1601,5588,32,0.71,23,1.44
8,4390,1601,6985,48,1.09,40,2.50
10,4304,1622,8174,157,3.65,122,7.52
12,4276,1619,8254,376,8.79,248,15.32
14,4255,1632,8254,366,8.60,262,16.05
